# Google Reviews Scraper

This notebook scrapes Google reviews for products using Selenium WebDriver. It follows the specified task flow to extract review text, star ratings, and dates.


In [40]:
# Import required libraries
import time
import random
import csv
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import warnings
warnings.filterwarnings('ignore')


In [41]:
# Step 1: Instantiate Selenium WebDriver
def setup_driver():
    """Set up Chrome WebDriver with appropriate options"""
    chrome_options = Options()
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option('useAutomationExtension', False)
    
    # Uncomment the line below if you want to run in headless mode
    # chrome_options.add_argument("--headless")
    
    driver = webdriver.Chrome(options=chrome_options)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    
    return driver

# Initialize the driver
driver = setup_driver()
print("Selenium WebDriver initialized successfully!")


Selenium WebDriver initialized successfully!


In [42]:
# Step 2 & 3: Open Google search and search for product
product = "iphone 17 pro max"  # Product variable as specified

def search_product(driver, product_name):
    """Open Google and search for the product"""
    # Navigate to Google
    driver.get("https://www.google.com")
    
    # Wait for search box and enter product name
    search_box = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.NAME, "q"))
    )
    
    search_box.clear()
    search_box.send_keys(product_name+" google reviews")
    search_box.submit()
    
    print(f"Searching for: {product_name} ")
    return driver

# Perform the search
driver = search_product(driver, product)


Searching for: iphone 17 pro max 


In [43]:
# Step 4: Find and click "More reviews" button
def find_and_click_more_reviews(driver):
    """Find and click the 'More reviews' button in the User Reviews section"""
    try:
        # Wait a bit for the page to load completely
        time.sleep(3)
        
        # Look for various possible selectors for the "More reviews" button
        possible_selectors = [
            "//span[contains(text(), 'More reviews')]",
            "//button[contains(text(), 'More reviews')]",
            "//div[contains(text(), 'More reviews')]",
            "//a[contains(text(), 'More reviews')]",
            "//span[contains(text(), 'View all reviews')]",
            "//button[contains(text(), 'View all reviews')]"
        ]
        
        for selector in possible_selectors:
            try:
                more_reviews_button = WebDriverWait(driver, 5).until(
                    EC.element_to_be_clickable((By.XPATH, selector))
                )
                driver.execute_script("arguments[0].click();", more_reviews_button)
                print("Successfully clicked 'More reviews' button")
                return True
            except TimeoutException:
                continue
        
        print("Could not find 'More reviews' button")
        return False
        
    except Exception as e:
        print(f"Error clicking 'More reviews' button: {e}")
        return False

# Click the "More reviews" button
more_reviews_clicked = find_and_click_more_reviews(driver)


Successfully clicked 'More reviews' button


In [44]:
# Step 5: Scrape reviews (text, stars, date)
def scrape_reviews_from_page(driver):
    """Scrape reviews from the current page"""
    reviews = []
    
    try:
        # Wait for reviews to load
        time.sleep(2)
        
        # Use the specific XPath structure you provided
        # Look for review containers using the jsname attribute
        review_selectors = [
            "//div[@jsname='I1taC']",
            "//div[contains(@class, 'TqzWd')]",
            "//div[contains(@data-ved, '')]"
        ]
        
        review_elements = []
        for selector in review_selectors:
            try:
                elements = driver.find_elements(By.XPATH, selector)
                if elements:
                    review_elements = elements
                    print(f"Found {len(elements)} reviews using selector: {selector}")
                    break
            except:
                continue
        
        if not review_elements:
            print("No review elements found")
            return reviews
        
        for i, review_element in enumerate(review_elements):
            try:
                review_data = {}
                
                # Extract review text - look for the full review text in the YdBXLd class
                text_selectors = [
                    ".//p[@class='YdBXLd']",
                    ".//p[contains(@class, 'YdBXLd')]",
                    ".//div[@jsname='QFmKA']//p",
                    ".//div[@jsname='an9Zef']//p"
                ]
                
                review_text = ""
                for text_selector in text_selectors:
                    try:
                        text_element = review_element.find_element(By.XPATH, text_selector)
                        review_text = text_element.text.strip()
                        if review_text:
                            break
                    except:
                        continue
                
                # Extract star rating - look for the aria-label with rating info
                star_selectors = [
                    ".//span[@class='z3HNkc']",
                    ".//span[contains(@aria-label, 'Rated')]",
                    ".//span[contains(@aria-label, 'out of 5')]"
                ]
                
                stars = ""
                for star_selector in star_selectors:
                    try:
                        star_element = review_element.find_element(By.XPATH, star_selector)
                        stars = star_element.get_attribute('aria-label') or star_element.text.strip()
                        if stars:
                            break
                    except:
                        continue
                
                # Extract date - look for the date span
                date_selectors = [
                    ".//span[@class='C8sUec']",
                    ".//span[contains(@aria-label, 'Review submitted')]",
                    ".//span[contains(@aria-label, 'ago')]"
                ]
                
                date = ""
                for date_selector in date_selectors:
                    try:
                        date_element = review_element.find_element(By.XPATH, date_selector)
                        date = date_element.get_attribute('aria-label') or date_element.text.strip()
                        if date:
                            break
                    except:
                        continue
                
                # Only add if we have at least the review text
                if review_text:
                    review_data = {
                        'review_text': review_text,
                        'stars': stars,
                        'date': date
                    }
                    reviews.append(review_data)
                    print(f"Scraped review {i+1}: {review_text[:50]}...")
                
            except Exception as e:
                print(f"Error scraping review {i+1}: {e}")
                continue
        
        print(f"Successfully scraped {len(reviews)} reviews from current page")
        
    except Exception as e:
        print(f"Error in scrape_reviews_from_page: {e}")
    
    return reviews

# Scrape reviews from the current page
current_reviews = scrape_reviews_from_page(driver)
print(f"Total reviews scraped: {len(current_reviews)}")


Found 11 reviews using selector: //div[@jsname='I1taC']
Scraped review 2: I hope Apple does not go pear-shaped with this rev...
Scraped review 3: Amazing cellphone! First, the orange color is spot...
Scraped review 4: I love the new iPhone 17 Pro Max, it has been a pl...
Scraped review 5: I just got it today. Very nice design and feel wit...
Scraped review 6: I found the device to not be much of drastic chang...
Scraped review 7: Very much enjoying my new iPhone 17 Pro Max. Loade...
Scraped review 8: The iPhone 17 Pro Max just feels different the mom...
Scraped review 9: The I phone 17 pro max is a very good mobile phone...
Scraped review 10: This is the most difficult phone to install that I...
Scraped review 11: People are extremely picky when it comes to a iPho...
Successfully scraped 10 reviews from current page
Total reviews scraped: 10


In [45]:
# Step 6: Handle pagination and scrape multiple pages
def scrape_all_reviews(driver, max_pages=5000):
    """Scrape reviews from multiple pages with pagination"""
    all_reviews = []
    page_count = 0
    consecutive_failures = 0
    max_consecutive_failures = 3
    
    while page_count < max_pages:
        print(f"\n--- Scraping page {page_count + 1} ---")
        
        # Scrape reviews from current page
        page_reviews = scrape_reviews_from_page(driver)
        all_reviews.extend(page_reviews)
        
        print(f"Total reviews collected so far: {len(all_reviews)}")
        
        # Step 7: Random timeout to avoid rate limiting
        timeout = random.randint(3000, 10000) / 1000  # Convert to seconds
        print(f"Waiting {timeout:.1f} seconds to avoid rate limiting...")
        time.sleep(timeout)
        
        # Try to find and click "More Reviews" for next page
        try:
            # Scroll down to ensure the "More Reviews" button is in viewport
            print("Scrolling down to find pagination button...")
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)  # Wait for scroll to complete
            
            # Use the specific XPath structure you provided
            pagination_selectors = [
                "/html/body/div[19]/div[2]/div[2]/div/div[2]/div[2]/button",
                "//div[2]/div[2]/button",
                "//button[contains(@class, '')]",
                "//span[contains(text(), 'More reviews')]",
                "//button[contains(text(), 'More reviews')]",
                "//div[contains(text(), 'More reviews')]",
                "//a[contains(text(), 'More reviews')]",
                "//span[contains(text(), 'Show more')]",
                "//button[contains(text(), 'Show more')]",
                "//div[contains(text(), 'Show more')]",
                "//a[contains(text(), 'Show more')]",
                "//span[contains(text(), 'Load more')]",
                "//button[contains(text(), 'Load more')]"
            ]
            
            next_page_found = False
            for i, selector in enumerate(pagination_selectors):
                try:
                    print(f"Trying pagination selector {i+1}: {selector}")
                    next_button = WebDriverWait(driver, 3).until(
                        EC.element_to_be_clickable((By.XPATH, selector))
                    )
                    
                    # Scroll to the button to ensure it's visible
                    driver.execute_script("arguments[0].scrollIntoView(true);", next_button)
                    time.sleep(1)
                    
                    driver.execute_script("arguments[0].click();", next_button)
                    print("Successfully clicked pagination button")
                    next_page_found = True
                    consecutive_failures = 0  # Reset failure counter
                    break
                except TimeoutException:
                    print(f"Selector {i+1} failed - button not found or not clickable")
                    continue
                except Exception as e:
                    print(f"Selector {i+1} failed with error: {e}")
                    continue
            
            if not next_page_found:
                consecutive_failures += 1
                print(f"No pagination button found (attempt {consecutive_failures}/{max_consecutive_failures})")
                
                if consecutive_failures >= max_consecutive_failures:
                    print("Reached maximum consecutive failures, stopping pagination")
                    break
                else:
                    # Try scrolling more and waiting longer
                    print("Trying to scroll more and wait longer...")
                    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                    time.sleep(5)
                    
                    # Try one more time with longer wait
                    for selector in pagination_selectors[:3]:  # Try only the most specific selectors
                        try:
                            next_button = WebDriverWait(driver, 5).until(
                                EC.element_to_be_clickable((By.XPATH, selector))
                            )
                            driver.execute_script("arguments[0].scrollIntoView(true);", next_button)
                            time.sleep(1)
                            driver.execute_script("arguments[0].click();", next_button)
                            print("Successfully clicked pagination button after extended wait")
                            next_page_found = True
                            consecutive_failures = 0
                            break
                        except:
                            continue
                    
                    if not next_page_found:
                        print("Still no pagination button found after extended wait")
                        continue
                
        except Exception as e:
            print(f"Error finding next page: {e}")
            consecutive_failures += 1
            if consecutive_failures >= max_consecutive_failures:
                break
        
        page_count += 1
    
    print(f"\nTotal reviews scraped across {page_count} pages: {len(all_reviews)}")
    return all_reviews

# Scrape reviews from multiple pages
all_reviews = scrape_all_reviews(driver, max_pages=5000)  # Increased from default 5 to 20



--- Scraping page 1 ---
Found 11 reviews using selector: //div[@jsname='I1taC']
Scraped review 2: I hope Apple does not go pear-shaped with this rev...
Scraped review 3: Amazing cellphone! First, the orange color is spot...
Scraped review 4: I love the new iPhone 17 Pro Max, it has been a pl...
Scraped review 5: I just got it today. Very nice design and feel wit...
Scraped review 6: I found the device to not be much of drastic chang...
Scraped review 7: Very much enjoying my new iPhone 17 Pro Max. Loade...
Scraped review 8: The iPhone 17 Pro Max just feels different the mom...
Scraped review 9: The I phone 17 pro max is a very good mobile phone...
Scraped review 10: This is the most difficult phone to install that I...
Scraped review 11: People are extremely picky when it comes to a iPho...
Successfully scraped 10 reviews from current page
Total reviews collected so far: 10
Waiting 3.4 seconds to avoid rate limiting...
Scrolling down to find pagination button...
Trying pagination sele

In [54]:
len(all_reviews)
all_reviews[2]

{'review_text': 'I love the new iPhone 17 Pro Max, it has been a pleasant experience upgrading from my iPhone 15 Pro Max. There isn’t a laundry list of differences, but it feels well done and offers a polished experience. The camera looks great, battery life has been good, and the design makes the phone much more comfortable to hold (even with a case). I am happy with the design refresh, as it is the only real refresh we’ve had since the 11 Pro series, and the aluminum paired with the vapor chamber has made this phone run much cooler and smoother. Overall I am very pleased with this new generation of iPhone, and I have no regrets upgrading.',
 'stars': 'Rated 5.0 out of 5,',
 'date': 'Review submitted a week ago'}

In [ ]:
# Step 8: Export scraped data to CSV
def save_reviews_to_csv(reviews, filename="google_reviews.csv"):
    """Save scraped reviews to CSV file"""
    if not reviews:
        print("No reviews to save")
        return
    
    # Create DataFrame
    df = pd.DataFrame(reviews)
    
    # Save to CSV
    df.to_csv(filename, index=False, encoding='utf-8')
    print(f"Successfully saved {len(reviews)} reviews to {filename}")
    
    # Display first few reviews
    print("\nFirst 5 reviews:")
    print(df.head())
    
    return df

# Save reviews to CSV
if all_reviews:
    df = save_reviews_to_csv(all_reviews, f"{product.replace(' ', '_')}_reviews.csv")
else:
    print("No reviews were scraped. Please check the selectors and try again.")


In [ ]:
# Clean up - Close the browser
def cleanup(driver):
    """Close the browser and clean up resources"""
    try:
        driver.quit()
        print("Browser closed successfully")
    except Exception as e:
        print(f"Error closing browser: {e}")

# Uncomment the line below when you're done scraping
# cleanup(driver)


## Usage Instructions

1. **Install required packages**: Make sure you have the following packages installed:
   ```bash
   pip install selenium pandas
   ```

2. **Chrome WebDriver**: Ensure you have Chrome browser installed and ChromeDriver available in your PATH, or install it using:
   ```bash
   pip install webdriver-manager
   ```

3. **Run the cells**: Execute the cells in order to scrape Google reviews.

4. **Customize**: You can modify the `product` variable to search for different products.

5. **Adjust parameters**: 
   - Change `max_pages` in the `scrape_all_reviews()` function to scrape more/fewer pages
   - Modify timeout ranges in the random sleep function
   - Uncomment the headless mode option if you want to run without opening a browser window

## Notes

- The scraper uses multiple selectors to handle different Google review layouts
- Random timeouts between 3-10 seconds help avoid rate limiting
- Reviews are saved to a CSV file with the product name
- The scraper handles pagination automatically
- Make sure to close the browser when done by uncommenting the cleanup function
